# Wadjet — Landmark Classifier (EfficientNet-B0 → ONNX)

**Architecture**: EfficientNet-B0 (timm) → 52 Egyptian landmark classes
**Input**: [1, 3, 224, 224] NCHW float32, values in [0.0, 1.0]
**Output**: [1, 52] softmax probabilities
**Training**: 2-phase (head → fine-tune), CrossEntropy + class weights
**Export**: torch.onnx.export → quantize_dynamic → uint8 ONNX

**CRITICAL RULES**:
- float32 ONLY — NO mixed precision (causes fused ops that break ONNX)
- Normalization: /255 only (NOT ImageNet mean/std) — must match inference code
- HorizontalFlip allowed (real-world photos, not direction-sensitive)
- Data is pre-merged + augmented (v1 splits + eg-landmarks + augment to min=300)

In [ ]:
# lightning is pre-installed on Kaggle as pytorch_lightning
# onnxscript is needed by PyTorch 2.10+ for torch.onnx.export
# onnxruntime is needed for ONNX validation and quantization
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'onnxscript', 'onnxruntime'])

In [ ]:
import os, json, math, time, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import timm
import pytorch_lightning as L
from sklearn.metrics import classification_report, top_k_accuracy_score
import albumentations as A
from albumentations.pytorch import ToTensorV2
from collections import Counter

warnings.filterwarnings('ignore', category=UserWarning)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Configuration ────────────────────────────────────────────────
INPUT_SIZE = 224
NUM_CLASSES = 52
BATCH_SIZE = 32
NUM_WORKERS = 2

# Phase 1: head only
P1_EPOCHS = 8
P1_LR = 1e-3

# Phase 2: fine-tune top 50%
P2_EPOCHS = 40
P2_LR = 5e-5

LABEL_SMOOTHING = 0.1

# Kaggle dataset paths — auto-discover mount point
OUT_DIR   = "/kaggle/working"

# Discover the actual dataset mount path
print("\n── /kaggle/input/ contents ──")
for name in sorted(os.listdir("/kaggle/input")):
    full = os.path.join("/kaggle/input", name)
    tag = "DIR" if os.path.isdir(full) else "FILE"
    print(f"  {tag}: {name}")

# Search for train directory
DATA_ROOT = None
for name in os.listdir("/kaggle/input"):
    candidate = os.path.join("/kaggle/input", name)
    if os.path.isdir(candidate):
        children = os.listdir(candidate)
        if "train" in children:
            DATA_ROOT = candidate
            break
        for child in children:
            sub = os.path.join(candidate, child)
            if os.path.isdir(sub) and "train" in os.listdir(sub):
                DATA_ROOT = sub
                break
        if DATA_ROOT:
            break

if DATA_ROOT is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train" in dirs and "val" in dirs:
            DATA_ROOT = root
            break

if DATA_ROOT is None:
    raise FileNotFoundError("Could not find train/val/test dirs under /kaggle/input/")

print(f"\nDATA_ROOT = {DATA_ROOT}")
print(f"Contents: {sorted(os.listdir(DATA_ROOT))}")

TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"\n!! float32 ONLY -- no mixed precision!")
print(f"!! Normalization: /255 only -- NO ImageNet mean/std!")
print(f"!! HorizontalFlip: YES (real-world photos)")

In [ ]:
# ── Verify dataset exists ────────────────────────────────────────
for split, path in [("train", TRAIN_DIR), ("val", VAL_DIR), ("test", TEST_DIR)]:
    if os.path.exists(path):
        classes = sorted(os.listdir(path))
        total = sum(len(os.listdir(os.path.join(path, c))) for c in classes)
        print(f"{split:5s}: {total:>6,} images, {len(classes)} classes")
    else:
        print(f"ERROR: {path} not found!")
        raise FileNotFoundError(path)

In [ ]:
# ── Albumentations transforms ────────────────────────────────────
# HorizontalFlip allowed for landmarks (real-world photos, not direction-sensitive)
# CRITICAL: Normalize to [0,1] only (/255) — must match inference code.
#           Backend does: np.array(img, float32) / 255.0

class AlbumentationsTransform:
    def __init__(self, transform):
        self.transform = transform
    def __call__(self, img):
        img_np = np.array(img)
        return self.transform(image=img_np)["image"]

train_transform = AlbumentationsTransform(A.Compose([
    A.Resize(INPUT_SIZE, INPUT_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, border_mode=0, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05, p=0.3),
    A.RandomShadow(shadow_roi=(0, 0.5, 1, 1), p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    # Normalize to [0,1] — divide by 255, no mean/std shift
    A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0), max_pixel_value=255.0),
    ToTensorV2(),
]))

val_transform = AlbumentationsTransform(A.Compose([
    A.Resize(INPUT_SIZE, INPUT_SIZE),
    A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0), max_pixel_value=255.0),
    ToTensorV2(),
]))

print("Transforms ready")
print("  - HorizontalFlip: YES")
print("  - Normalization: /255 to [0,1] (no ImageNet mean/std)")

In [ ]:
# ── Datasets & DataLoaders ───────────────────────────────────────
train_ds = ImageFolder(TRAIN_DIR, transform=train_transform)
val_ds   = ImageFolder(VAL_DIR,   transform=val_transform)
test_ds  = ImageFolder(TEST_DIR,  transform=val_transform)

# Safety: remap val/test to use train's class_to_idx (prevents label misalignment
# if any class is missing from a split)
for ds, name in [(val_ds, 'val'), (test_ds, 'test')]:
    old_classes = ds.classes
    if set(old_classes) != set(train_ds.classes):
        missing = set(train_ds.classes) - set(old_classes)
        print(f"  {name}: remapping {len(old_classes)} -> {len(train_ds.classes)} classes (missing: {sorted(missing)})")
    ds.class_to_idx = train_ds.class_to_idx
    ds.classes = train_ds.classes
    new_samples = []
    for path, _old_idx in ds.samples:
        class_name = os.path.basename(os.path.dirname(path))
        new_idx = train_ds.class_to_idx.get(class_name)
        if new_idx is not None:
            new_samples.append((path, new_idx))
    ds.samples = new_samples
    ds.targets = [s[1] for s in new_samples]
    ds.imgs = new_samples

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

class_names = train_ds.classes
assert len(class_names) == NUM_CLASSES, f"Expected {NUM_CLASSES} classes, got {len(class_names)}"

print(f"Train: {len(train_ds):,} images, {len(class_names)} classes")
print(f"Val:   {len(val_ds):,} images")
print(f"Test:  {len(test_ds):,} images")
print(f"First 5 classes: {class_names[:5]}")
print(f"Last 5 classes:  {class_names[-5:]}")

sample, label = train_ds[0]
print(f"\nSample tensor: shape={sample.shape}, min={sample.min():.3f}, max={sample.max():.3f}")
assert sample.shape == (3, INPUT_SIZE, INPUT_SIZE), f"Wrong shape: {sample.shape}"
assert sample.min() >= 0.0 and sample.max() <= 1.0, f"Values must be in [0,1]!"
print("Tensor assertions passed")

In [ ]:
# ── Class weights (sqrt-inverse-frequency) ──────────────────────
class_counts = Counter()
for _, label in train_ds.samples:
    class_counts[label] += 1

total = sum(class_counts.values())
weights = []
for i in range(NUM_CLASSES):
    count = class_counts.get(i, 1)
    w = math.sqrt(total / (NUM_CLASSES * count))
    weights.append(w)

class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
print(f"Weight range: {class_weights.min():.3f} -- {class_weights.max():.3f}")
print(f"Most common class ({class_names[class_counts.most_common(1)[0][0]]}): {class_counts.most_common(1)[0][1]} imgs")
print(f"Least common class: {class_counts.most_common()[-1][1]} imgs")

In [ ]:
# ── Loss: CrossEntropy + class weights + label smoothing ─────────
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=LABEL_SMOOTHING
)
print("CrossEntropyLoss ready")

In [ ]:
# ── Lightning Module ─────────────────────────────────────────────
class LandmarkClassifier(L.LightningModule):
    def __init__(self, num_classes, lr, criterion, freeze_backbone=False):
        super().__init__()
        self.save_hyperparameters(ignore=['criterion'])
        self.criterion = criterion
        self.lr = lr

        # EfficientNet-B0 with ImageNet pretrained weights
        self.model = timm.create_model('efficientnet_b0',
                                       pretrained=True, num_classes=num_classes)
        if freeze_backbone:
            self._freeze_backbone()

    def _freeze_backbone(self):
        for name, param in self.model.named_parameters():
            if 'classifier' not in name:
                param.requires_grad = False
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.model.parameters())
        print(f"Frozen: {trainable:,}/{total:,} params trainable ({100*trainable/total:.1f}%)")

    def unfreeze_top(self, fraction=0.5):
        """Unfreeze the top `fraction` of backbone layers."""
        params = [(n, p) for n, p in self.model.named_parameters()
                  if 'classifier' not in n]
        n_unfreeze = int(len(params) * fraction)
        for name, param in params[-n_unfreeze:]:
            param.requires_grad = True
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.model.parameters())
        print(f"Unfroze top {fraction*100:.0f}%: {trainable:,}/{total:,} params trainable ({100*trainable/total:.1f}%)")

    def forward(self, x):
        return self.model(x)

    def _shared_step(self, batch, stage):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()
        self.log(f"{stage}_loss", loss, prog_bar=True)
        self.log(f"{stage}_acc", acc, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, "val")

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, self.parameters()),
            lr=self.lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.trainer.max_epochs)
        return [optimizer], [scheduler]

print("LandmarkClassifier defined")

# Prevent Kaggle/papermill IOPub timeout (4s) during long training
class KeepAliveCallback(L.Callback):
    """Prints dots every N batches to keep IOPub alive."""
    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        if batch_idx % 10 == 0:
            print(".", end="", flush=True)
    def on_validation_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        acc = metrics.get("val_acc", 0)
        print(f" val_acc={acc:.4f}", flush=True)

print("KeepAliveCallback defined")

In [ ]:
# ── Phase 1: Head Training (backbone frozen) ─────────────────────
# CRITICAL: precision="32-true" — NO mixed precision!
# Mixed precision creates fused ops that can break ONNX export.

model = LandmarkClassifier(
    num_classes=NUM_CLASSES, lr=P1_LR,
    criterion=criterion, freeze_backbone=True
)

trainer_p1 = L.Trainer(
    max_epochs=P1_EPOCHS,
    accelerator="auto",
    precision="32-true",  # float32 ONLY — no mixed precision!
    callbacks=[
        L.callbacks.ModelCheckpoint(
            dirpath=os.path.join(OUT_DIR, "checkpoints"),
            monitor="val_acc", mode="max", save_top_k=1,
            filename="best-p1-{epoch}-{val_acc:.4f}"),
        KeepAliveCallback(),
    ],
    logger=False,
    enable_progress_bar=False,
)

print("=" * 60)
print("Phase 1: Head training (backbone frozen, float32)")
print("=" * 60)
t0 = time.time()
trainer_p1.fit(model, train_loader, val_loader)
print(f"Phase 1 done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── Phase 2: Fine-tune top 50% ──────────────────────────────────
model.unfreeze_top(fraction=0.5)
model.lr = P2_LR

trainer_p2 = L.Trainer(
    max_epochs=P2_EPOCHS,
    accelerator="auto",
    precision="32-true",  # float32 ONLY — no mixed precision!
    callbacks=[
        L.callbacks.ModelCheckpoint(
            dirpath=os.path.join(OUT_DIR, "checkpoints"),
            monitor="val_acc", mode="max", save_top_k=1,
            filename="best-p2-{epoch}-{val_acc:.4f}"),
        L.callbacks.EarlyStopping(
            monitor="val_acc", patience=7, mode="max"),
        KeepAliveCallback(),
    ],
    logger=False,
    enable_progress_bar=False,
)

print("=" * 60)
print("Phase 2: Fine-tuning top 50% of backbone (float32)")
print("=" * 60)
t0 = time.time()
trainer_p2.fit(model, train_loader, val_loader)
print(f"Phase 2 done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ── Evaluation on test set ───────────────────────────────────────
model.eval()
model.to(DEVICE)

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        logits = model(x)
        probs = F.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())
        all_probs.extend(probs)

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)
all_preds = np.array(all_preds)

top1_acc = np.mean(all_preds == all_labels)
top3_acc = top_k_accuracy_score(all_labels, all_probs, k=3,
                                 labels=list(range(NUM_CLASSES)))

print(f"\n{'='*60}")
print(f"Test Top-1 Accuracy: {top1_acc:.4f} {'PASS' if top1_acc >= 0.75 else 'BELOW 75% GATE'}")
print(f"Test Top-3 Accuracy: {top3_acc:.4f}")
print(f"{'='*60}")

# Per-class report
report = classification_report(all_labels, all_preds, labels=list(range(NUM_CLASSES)),
                               target_names=class_names, output_dict=True)
print(f"\nMacro F1: {report['macro avg']['f1-score']:.4f}")
print(f"Weighted F1: {report['weighted avg']['f1-score']:.4f}")

# Show worst 10 classes
class_f1 = [(name, report[name]['f1-score'], report[name]['support'])
            for name in class_names if name in report]
class_f1.sort(key=lambda x: x[1])
print(f"\nWorst 10 classes by F1:")
for name, f1, sup in class_f1[:10]:
    print(f"  {name:30s}  F1={f1:.3f}  support={int(sup)}")

In [ ]:
# ── ONNX Export (float32, NCHW, opset 18) ────────────────────────
model.eval()
model.to("cpu")

dummy = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE)
onnx_fp32_path = os.path.join(OUT_DIR, "landmark_classifier.onnx")

# Export with opset 18 (PyTorch 2.10 minimum)
# dynamo=False → legacy exporter, keeps weights internal (no .onnx.data file)
torch.onnx.export(
    model.model, dummy, onnx_fp32_path,
    opset_version=18,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    dynamo=False,
)

size_mb = os.path.getsize(onnx_fp32_path) / 1e6
print(f"ONNX fp32 exported: {onnx_fp32_path}")
print(f"Size: {size_mb:.1f} MB")

In [ ]:
# ── ONNX Validation (fp32) ───────────────────────────────────────
import onnxruntime as ort

sess = ort.InferenceSession(onnx_fp32_path)
input_info = sess.get_inputs()[0]
output_info = sess.get_outputs()[0]
print(f"Input:  name={input_info.name}, shape={input_info.shape}, type={input_info.type}")
print(f"Output: name={output_info.name}, shape={output_info.shape}, type={output_info.type}")

# Run inference with random data in [0,1] range
dummy_np = np.random.rand(1, 3, INPUT_SIZE, INPUT_SIZE).astype(np.float32)
out = sess.run(None, {"input": dummy_np})
assert out[0].shape == (1, NUM_CLASSES), f"Wrong shape: {out[0].shape}"
print(f"\nOutput shape: {out[0].shape}")
print(f"ONNX fp32 validation passed")

In [ ]:
# ── ONNX Quantization (uint8) ────────────────────────────────────
from onnxruntime.quantization import quantize_dynamic, QuantType

uint8_path = os.path.join(OUT_DIR, "landmark_classifier_uint8.onnx")
quantize_dynamic(onnx_fp32_path, uint8_path, weight_type=QuantType.QUInt8)

uint8_size_mb = os.path.getsize(uint8_path) / 1e6
print(f"Quantized: {uint8_path}")
print(f"Size: {uint8_size_mb:.1f} MB (was {size_mb:.1f} MB fp32)")
print(f"Compression: {size_mb/uint8_size_mb:.1f}x")

# Validate quantized model
sess_q = ort.InferenceSession(uint8_path)
out_q = sess_q.run(None, {"input": dummy_np})
assert out_q[0].shape == (1, NUM_CLASSES), f"Wrong shape: {out_q[0].shape}"
print(f"Quantized output shape: {out_q[0].shape}")

In [ ]:
# ── Save label mapping + metadata ────────────────────────────────
label_mapping = {i: name for i, name in enumerate(class_names)}
with open(os.path.join(OUT_DIR, "landmark_label_mapping.json"), "w") as f:
    json.dump(label_mapping, f, indent=2)

metadata = {
    "model": "efficientnet_b0",
    "framework": "pytorch",
    "input_size": INPUT_SIZE,
    "num_classes": NUM_CLASSES,
    "format": "NCHW",
    "input_range": [0.0, 1.0],
    "normalization": "divide_by_255",
    "test_top1_accuracy": float(top1_acc),
    "test_top3_accuracy": float(top3_acc),
    "quantized": True,
    "opset_version": 18,
    "fp32_size_mb": round(size_mb, 1),
    "uint8_size_mb": round(uint8_size_mb, 1),
    "training": {
        "phase1_epochs": P1_EPOCHS,
        "phase2_epochs": P2_EPOCHS,
        "loss": f"CrossEntropyLoss(smoothing={LABEL_SMOOTHING})",
        "augmentation": "h-flip+rotation_15deg+brightness+contrast+shadow+blur",
        "precision": "float32 (no mixed precision)"
    }
}
with open(os.path.join(OUT_DIR, "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved: landmark_label_mapping.json")
print(f"  {NUM_CLASSES} classes: {class_names[0]} -> {class_names[-1]}")
print("Saved: model_metadata.json")
print(f"  normalization: divide_by_255 (input range [0,1])")
print(f"  format: NCHW [{1},{3},{INPUT_SIZE},{INPUT_SIZE}]")

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Test Top-1: {top1_acc:.4f} {'PASS' if top1_acc >= 0.75 else 'FAIL'}")
print(f"Test Top-3: {top3_acc:.4f}")
print(f"\nOutput files in {OUT_DIR}:")
for fname in sorted(os.listdir(OUT_DIR)):
    fpath = os.path.join(OUT_DIR, fname)
    if os.path.isfile(fpath):
        fsize = os.path.getsize(fpath)
        print(f"  {fname:50s} {fsize/1e6:>8.1f} MB")
print(f"\nDownload with:")
print(f"  kaggle kernels output nadermohamedcr7/wadjet-landmark-classifier -p models/landmark/classifier/")